# ⚡ WaveForge 3D — Kaggle 2x T4 GPU Benchmark

**Just click Run All — fully automated.**

This notebook benchmarks WaveForge 3D on **2x Tesla T4** GPUs (Kaggle accelerator):
- Single-GPU 3D scaling on GPU[0] and GPU[1] separately
- All 10 3D examples on GPU[0]
- Dual-GPU comparison (side-by-side GPU[0] vs GPU[1])
- Comparison vs pre-recorded CPU baseline

---
**Kaggle setup:** Accelerator → GPU → 2x T4 GPU  
Dataset: not needed — repo is cloned automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone repo + setup                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np, json, platform, datetime, re, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.ticker as mticker

assert torch.cuda.is_available(), 'No CUDA — enable 2x T4 GPU accelerator'
N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')
print(f'PyTorch {torch.__version__}  CUDA {torch.version.cuda}')
print('✅ Setup complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load CPU baselines                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def jload(p):
    with open(p) as f: return json.load(f)

cpu2d   = jload('benchmarks/cpu_results.json')          # 2D NxN CPU scaling
cpu3d   = jload('benchmarks/3d_examples_cpu_results.json')  # 3D examples CPU
meep    = jload('benchmarks/meep_comparison_results.json')  # PyMEEP 2D

print('Baselines:')
print(f'  [2D] WaveForge CPU scaling  : {len(cpu2d["waveforge_cpu"])} sizes')
print(f'  [3D] WaveForge CPU examples : {len(cpu3d)} examples')
print(f'  [2D] PyMEEP scenes          : {len(meep)} scenes')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — 3D Grid Scaling on each GPU independently                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from core.grid import YeeGrid
from core.fields import FieldSet
from core.boundaries import MurABC3D
from core.sources import GaussianPulse, PointSource, SourceCollection
from core.fdtd3d import FDTD3D

N_WARMUP   = 20
N_STEPS    = 100
GRID_SIZES = [32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024]

def bench3d(N, device, warmup=20, steps=100):
    DX = 1.5e-3
    grid     = YeeGrid(N, N, dx=DX, dy=DX, Nz=N, dz=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC3D(grid, fields.Hx, fields.Hy, fields.Hz)
    pulse    = GaussianPulse(1.0, sigma=20*grid.dt)
    cx=cy=cz = N//2
    src = PointSource(pulse, cx, cy, 'Ez', k=cz, grid=grid, N_steps=warmup+steps)
    sim = FDTD3D(grid, fields, boundary, SourceCollection([src]), n_check=999999)
    with torch.no_grad(): sim.run(warmup)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): sim.run(steps)
    torch.cuda.synchronize()
    el = time.perf_counter()-t0
    return round(steps*N**3/el/1e6, 1), round(el/steps*1e3, 3)

scaling_per_gpu = {}
for gpu_idx in range(N_GPUS):
    dev = f'cuda:{gpu_idx}'
    name = torch.cuda.get_device_name(gpu_idx)
    print(f'\nGPU[{gpu_idx}] — {name} on {dev}')
    print(f'{"N":>7}  {"Mcells/s":>10}  {"ms/step":>10}')
    print('-'*32)
    rows = []
    for N in GRID_SIZES:
        try:
            mc, ms = bench3d(N, dev, N_WARMUP, N_STEPS)
            rows.append({'N':N,'mcells_s':mc,'ms_step':ms})
            print(f'{N:7d}³  {mc:10.1f}  {ms:10.3f}')
        except (torch.cuda.OutOfMemoryError, RuntimeError):
            print(f'{N:7d}³  OOM')
            break
    scaling_per_gpu[gpu_idx] = rows

print('\n✅ Grid-scaling done on all GPUs')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Run all 10 3D examples on GPU[0]                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
os.makedirs('examples/output', exist_ok=True)
examples_gpu0 = []

for entry in cpu3d:
    fname = entry['file']
    print(f'  {fname}...', end=' ', flush=True)
    t0 = time.time()
    try:
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
        res = subprocess.run([sys.executable, f'examples/3d/{fname}'],
                             capture_output=True, text=True, timeout=360, env=env)
        el  = time.time()-t0
        out = res.stdout+res.stderr
        m   = re.search(r'WAVEFORGE_BENCH:\s*([\d.]+)', out)
        g   = re.search(r'Grid:\s*(\d+)x(\d+)x(\d+)', out)
        mc  = float(m.group(1)) if m else 0.0
        grd = f'{g.group(1)}x{g.group(2)}x{g.group(3)}' if g else entry['grid']
        ok  = res.returncode==0 and mc>0
        examples_gpu0.append({'file':fname,'status':'PASS' if ok else 'FAIL',
                               'time_s':round(el,1),'mcells_s':mc,'grid':grd})
        print(f'{'PASS' if ok else 'FAIL'} | {el:.1f}s | {mc:.1f} Mc/s')
        if not ok:
            for line in out.strip().split('\n')[-3:]: print(f'    {line}')
    except subprocess.TimeoutExpired:
        examples_gpu0.append({'file':fname,'status':'TIMEOUT','time_s':360,'mcells_s':0,'grid':''})
        print('TIMEOUT')

n_pass = sum(1 for r in examples_gpu0 if r['status']=='PASS')
print(f'\n✅ {n_pass}/{len(examples_gpu0)} examples passed on GPU[0]')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Save results JSON                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
results = {
    'meta': {
        'date'     : datetime.datetime.now().isoformat(),
        'platform' : 'Kaggle',
        'n_gpus'   : N_GPUS,
        'gpu_names': [torch.cuda.get_device_name(i) for i in range(N_GPUS)],
        'vram_gb'  : [round(torch.cuda.get_device_properties(i).total_memory/1e9,1) for i in range(N_GPUS)],
        'torch'    : torch.__version__,
        'cuda'     : torch.version.cuda,
        'python'   : platform.python_version(),
        'note'     : '3D benchmark NxNxN grids'
    },
    'grid_scaling_per_gpu': {str(k): v for k,v in scaling_per_gpu.items()},
    'examples_gpu0'       : examples_gpu0,
}
out_path = 'benchmarks/kaggle_3d_gpu_results.json'
with open(out_path,'w') as f: json.dump(results, f, indent=2)
print(f'Saved → {out_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Plot A: 3D Scaling — GPU[0] vs GPU[1] vs CPU                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
os.makedirs('docs/assets', exist_ok=True)
colors = ['crimson', 'darkorange', 'royalblue', 'green']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'WaveForge 3D — Kaggle {N_GPUS}x T4 Grid Scaling (NxNxN)',
             fontsize=14, fontweight='bold')

# Throughput
ax = axes[0]
cpu2d_N  = [r['N'] for r in cpu2d['waveforge_cpu']]
cpu2d_mc = [r['mcells_s'] for r in cpu2d['waveforge_cpu']]
ax.plot(cpu2d_N, cpu2d_mc, 'o--', color='royalblue', lw=2, label='WaveForge CPU (2D ref)')
ax.plot([r['N'] for r in cpu2d['meep_cpu']], [r['mcells_s'] for r in cpu2d['meep_cpu']],
        'x:', color='gray', lw=1.5, label='PyMEEP CPU (2D ref)')

for i, rows in scaling_per_gpu.items():
    name = torch.cuda.get_device_name(i)
    ax.plot([r['N'] for r in rows], [r['mcells_s'] for r in rows],
            's-' if i==0 else '^-', color=colors[i], lw=2.5,
            label=f'GPU[{i}] {name.split()[1] if len(name.split())>1 else name}')

ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('3D Throughput vs Grid Size')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x)}³'))

# Speedup vs CPU
ax2 = axes[1]
cpu_d = {r['N']: r['mcells_s'] for r in cpu2d['waveforge_cpu']}
for i, rows in scaling_per_gpu.items():
    name = torch.cuda.get_device_name(i)
    ns  = [r['N'] for r in rows if r['N'] in cpu_d]
    spu = [r['mcells_s']/cpu_d[r['N']] for r in rows if r['N'] in cpu_d]
    ax2.plot(ns, spu, 's-' if i==0 else '^-', color=colors[i], lw=2.5,
             label=f'GPU[{i}] / CPU')
ax2.axhline(1, color='royalblue', ls='--', lw=1.5, label='CPU (1×)')
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax2.set_ylabel('GPU / CPU speedup  (×)', fontsize=11)
ax2.set_title('GPU Speedup over CPU')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, which='both')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x)}³'))

plt.tight_layout()
plt.savefig('docs/assets/kaggle_3d_scaling.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved → docs/assets/kaggle_3d_scaling.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Plot B: All 10 3D Examples — CPU vs GPU[0]                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('WaveForge 3D — All 10 Examples on Kaggle T4', fontsize=14, fontweight='bold')

labels = [r['file'].replace('3d_','').replace('.py','') for r in cpu3d]
cpu_mc = [r['mcells_s'] for r in cpu3d]
gpu_mc = [r['mcells_s'] for r in examples_gpu0]
cpu_t  = [r['time_s']   for r in cpu3d]
gpu_t  = [r['time_s']   for r in examples_gpu0]
x = np.arange(len(labels)); w = 0.35

ax = axes[0]
ax.bar(x-w/2, cpu_mc, w, label='CPU (baseline)', color='royalblue', alpha=0.85)
ax.bar(x+w/2, gpu_mc, w, label=f'GPU[0] — {torch.cuda.get_device_name(0).split()[1]}',
       color='crimson', alpha=0.85)
for i,(c,g) in enumerate(zip(cpu_mc, gpu_mc)):
    if c>0 and g>0:
        ax.text(x[i]+w/2, g+0.4, f'{g/c:.1f}×', ha='center', fontsize=7,
                color='darkred', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('Throughput per 3D Example')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.bar(x-w/2, cpu_t, w, label='CPU', color='royalblue', alpha=0.85)
ax2.bar(x+w/2, gpu_t, w, label='GPU[0]', color='crimson', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax2.set_ylabel('Wall time (s)', fontsize=11)
ax2.set_title('Runtime per 3D Example')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/kaggle_3d_examples.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved → docs/assets/kaggle_3d_examples.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Full Summary Table                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
W = 74
def row(s): print(f'║  {s:<{W-4}}║')
def sep():  print('║'+'─'*W+'║')
def hdr():  print('╠'+'═'*W+'╣')

print('╔'+'═'*W+'╗')
row(f'WAVEFORGE 3D — KAGGLE {N_GPUS}x {torch.cuda.get_device_name(0)} BENCHMARK')

# Grid scaling
hdr(); row(f'3D GRID SCALING  ({N_WARMUP} warmup + {N_STEPS} steps, NxNxN grids)')
header = f'{"N":>8}  {"CPU (Mc/s)":>10}'
for i in range(N_GPUS): header += f'  {f"GPU[{i}] (Mc/s)":>13}  {f"GPU[{i}]/CPU":>10}'
row(header); sep()

cpu_d   = {r['N']: r['mcells_s'] for r in cpu2d['waveforge_cpu']}
gpu_ds  = [{r['N']: r['mcells_s'] for r in scaling_per_gpu[i]} for i in range(N_GPUS)]
all_Ns  = sorted(set(cpu_d) | set().union(*[set(d) for d in gpu_ds]))
for N in all_Ns:
    c = cpu_d.get(N,0)
    line = f'{N:8d}³  {c:10.1f}'
    for gd in gpu_ds:
        g = gd.get(N,0)
        sp = f'{g/c:.1f}×' if c>0 and g>0 else '-'
        line += f'  {g:13.1f}  {sp:>10}'
    row(line)

for i, rows in scaling_per_gpu.items():
    if rows:
        pN = max(rows, key=lambda r: r['mcells_s'])['N']
        pm = max(rows, key=lambda r: r['mcells_s'])['mcells_s']
        sep(); row(f'GPU[{i}] Peak: {pm:.1f} Mcells/s at {pN}³')

# Examples
hdr(); row('3D EXAMPLES  (GPU[0])')
row(f'{"Example":<32}  {"CPU (Mc/s)":>10}  {"GPU[0] (Mc/s)":>13}  {"Speedup":>8}  {"GPU time":>9}')
sep()
tot_c = tot_g = 0
for c,g in zip(cpu3d, examples_gpu0):
    name = c['file'].replace('3d_','').replace('.py','')
    sp   = f'{g["mcells_s"]/c["mcells_s"]:.1f}×' if c['mcells_s']>0 and g['mcells_s']>0 else '-'
    row(f'{name:<32}  {c["mcells_s"]:>10.1f}  {g["mcells_s"]:>13.1f}  {sp:>8}  {g["time_s"]:>7.1f}s')
    tot_c+=c['time_s']; tot_g+=g['time_s']
sep()
tsp = f'{tot_c/tot_g:.1f}×' if tot_g>0 else '-'
row(f'{"Total":<32}  {tot_c:>9.0f}s  {tot_g:>12.0f}s  {tsp:>8}')

print('╚'+'═'*W+'╝')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Copy outputs to /kaggle/working for download                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil
OUT_DIR = pathlib.Path('/kaggle/working/waveforge_results')
OUT_DIR.mkdir(exist_ok=True)

# Copy JSON
shutil.copy(out_path, OUT_DIR)

# Copy plots
for p in ['docs/assets/kaggle_3d_scaling.png',
          'docs/assets/kaggle_3d_examples.png']:
    if pathlib.Path(p).exists():
        shutil.copy(p, OUT_DIR)

# Copy 3D example PNGs
for p in pathlib.Path('examples/output').glob('3d_*.png'):
    shutil.copy(p, OUT_DIR)

print('Files in /kaggle/working/waveforge_results:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size//1024} KB)')

print('\n🏁 All done! Download from the Kaggle output panel.')